# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** TODO — replace with your domain

**User:** TODO — who will use this agent?

**Success looks like:** TODO — what does a good outcome mean?

**Tool I will add:** TODO — what tool beyond retrieval? (web search, calculator, date, domain-specific)

**Deployment choice:** TODO — Streamlit UI or FastAPI endpoint?

---
## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [ ]:
# TODO: Replace these with your domain documents
# Each document needs: id, topic, text
# Minimum 10 documents — add more for better coverage

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "TODO — topic name",
        "text": """TODO — write or paste your first document here.
This should be 100-300 words about one specific aspect of your domain.
Be specific — vague documents produce vague answers."""
    },
    {
        "id": "doc_002",
        "topic": "TODO — topic name",
        "text": """TODO — second document."""
    },
    # Add doc_003 through doc_010 (minimum)
    # Pattern: {"id": "doc_XXX", "topic": "...", "text": """..."""}
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

In [ ]:
# ── Test retrieval before building the graph ──────────────
# TODO: Replace with a question relevant to your domain
test_query = "TODO — write a test question from your domain"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [ ]:
# TODO: Extend this State with any domain-specific fields you need
# Examples:
#   quiz_score: int          — for a Study Buddy that tracks scores
#   code_to_review: str      — for a Code Review Agent
#   employee_id: str         — for an HR Policy Bot
#   search_results: str      — if you use web search tool

class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from tool call

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter

    # TODO: Add your domain-specific fields here
    # e.g. search_results: str

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [ ]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

In [ ]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides: retrieve, memory_only, or tool
# TODO: Customise the prompt to match your domain

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    # TODO: Update the domain description and tool description below
    prompt = f"""You are a router for a chatbot about TODO_YOUR_DOMAIN.

Available options:
- retrieve: search the knowledge base for domain-specific information
- memory_only: answer from conversation history (e.g. 'what did you just say?')
- tool: use the TODO_TOOL_NAME tool (e.g. TODO_WHEN_TO_USE_TOOL)

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    # Clean up LLM output
    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

In [ ]:
# ── Node 3: Retrieval ──────────────────────────────────────
# Queries ChromaDB — no changes needed

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "TODO — replace with a question from your domain"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

In [ ]:
# ── Node 4: Tool ───────────────────────────────────────────
# TODO: Replace this with your actual tool
# Examples from previous days:
#   Web search (Day 2):   from ddgs import DDGS
#   Calculator (Day 2):   eval(expression)
#   Date tool (Day 9):    datetime.date.today()
#   Weather (Day 9):      requests.get(weather_api)

def tool_node(state: CapstoneState) -> dict:
    """TODO: Replace with your actual tool logic."""
    question = state["question"]

    # TODO: Implement your tool here
    # Example — web search:
    # try:
    #     from ddgs import DDGS
    #     with DDGS() as ddgs:
    #         results = list(ddgs.text(question, max_results=3))
    #     tool_result = "\n".join(f"{r['title']}: {r['body'][:200]}" for r in results)
    # except Exception as e:
    #     tool_result = f"Search error: {e}"

    # Placeholder — remove when you add real tool logic
    tool_result = f"[Tool result for: {question}] — TODO: implement your tool here"

    return {"tool_result": tool_result}


print("tool_node defined — replace the placeholder with your real tool logic")

In [ ]:
# ── Node 5: Answer ─────────────────────────────────────────
# Combines memory + retrieved context + tool results → LLM answer
# TODO: Customise the system prompt for your domain

def answer_node(state: CapstoneState) -> dict:
    question    = state["question"]
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages    = state.get("messages", [])
    eval_retries= state.get("eval_retries", 0)

    # Build context section
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    # TODO: Replace the system prompt with one suited to your domain
    # Keep the grounding rule — it's what makes the agent faithful
    if context:
        system_content = f"""You are a helpful assistant for TODO_YOUR_DOMAIN.
Answer using ONLY the information provided in the context below.
If the answer is not in the context, say: I don't have that information in my knowledge base.
Do NOT add information from your training data.

{context}"""
    else:
        system_content = """You are a helpful assistant. Answer based on the conversation history."""

    # If this is a retry after eval failure, add improvement instruction
    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

    # Build message list: system + history + current question
    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined — update the system prompt for your domain")

In [ ]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [ ]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [ ]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


# TODO: Define your 10 test questions
# Include at least 2 red-team tests:
#   - One out-of-scope question (should admit it doesn't know)
#   - One adversarial question with a false premise (should correct it)

TEST_QUESTIONS = [
    # Domain questions (from your knowledge base)
    {"q": "TODO — question 1 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 2 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 3 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 4 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 5 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 6 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 7 from your domain",    "expect": "Should answer from KB",          "red_team": False},
    {"q": "TODO — question 8 (memory test)",       "expect": "Should reference earlier answer", "red_team": False},
    # Red-team tests
    {"q": "TODO — out-of-scope question (not in KB)", "expect": "Should admit it doesn't know", "red_team": True},
    {"q": "TODO — false premise question",            "expect": "Should correct the premise",   "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

In [ ]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # TODO: Judge each test as PASS or FAIL
    # Change the logic below to match your expected outcomes
    passed = len(answer) > 20  # placeholder — replace with real check

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

# Summary
total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

---
## Part 6 — RAGAS Baseline Evaluation

In [ ]:
# TODO: Add ground truth answers for your test questions
# These are the correct answers you expect the agent to give

RAGAS_QUESTIONS = [
    {"question": "TODO — question 1", "ground_truth": "TODO — correct answer 1"},
    {"question": "TODO — question 2", "ground_truth": "TODO — correct answer 2"},
    {"question": "TODO — question 3", "ground_truth": "TODO — correct answer 3"},
    {"question": "TODO — question 4", "ground_truth": "TODO — correct answer 4"},
    {"question": "TODO — question 5", "ground_truth": "TODO — correct answer 5"},
]

# Build the eval dataset
eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

In [ ]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [ ]:
# TODO: Update DOMAIN_NAME and DOMAIN_DESCRIPTION
DOMAIN_NAME        = "TODO — Your Domain Name"
DOMAIN_DESCRIPTION = "TODO — One sentence description of what your agent does"
KB_TOPICS          = [d["topic"] for d in DOCUMENTS]

capstone_streamlit = f'''
"""
capstone_streamlit.py — {DOMAIN_NAME} Agent
Run: streamlit run capstone_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(page_title="{DOMAIN_NAME}", page_icon="🤖", layout="centered")
st.title("🤖 {DOMAIN_NAME}")
st.caption("{DOMAIN_DESCRIPTION}")

# ── Load models and KB (cached) ───────────────────────────
@st.cache_resource
def load_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try: client.delete_collection("capstone_kb")
    except: pass
    collection = client.create_collection("capstone_kb")

    # TODO: Copy your DOCUMENTS list here
    DOCUMENTS = [
        {{"id":"doc_001", "topic":"TODO", "text":"TODO — paste your documents here"}},
    ]
    texts = [d["text"] for d in DOCUMENTS]
    collection.add(documents=texts, embeddings=embedder.encode(texts).tolist(),
                   ids=[d["id"] for d in DOCUMENTS],
                   metadatas=[{{"topic":d["topic"]}} for d in DOCUMENTS])

    # TODO: Copy your CapstoneState, node functions, and graph assembly here
    # ... (paste from notebook Parts 2-4)

    return agent_app, embedder, collection


try:
    agent_app, embedder, collection = load_agent()
    st.success(f"✅ Knowledge base loaded — {{collection.count()}} documents")
except Exception as e:
    st.error(f"Failed to load agent: {{e}}")
    st.stop()

# ── Session state ─────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("About")
    st.write("{DOMAIN_DESCRIPTION}")
    st.write(f"Session: {{st.session_state.thread_id}}")
    st.divider()
    st.write("**Topics covered:**")
    for t in {KB_TOPICS}:
        st.write(f"• {{t}}")
    if st.button("🗑️ New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

# ── Display history ───────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask something..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({{"role":"user","content":prompt}})

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            config = {{"configurable": {{"thread_id": st.session_state.thread_id}}}}
            result = agent_app.invoke({{"question": prompt}}, config=config)
            answer = result.get("answer", "Sorry, I could not generate an answer.")
        st.write(answer)
        faith = result.get("faithfulness", 0.0)
        if faith > 0:
            st.caption(f"Faithfulness: {{faith:.2f}} | Sources: {{result.get(\'sources\', [])}}") 

    st.session_state.messages.append({{"role":"assistant","content":answer}})
'''

with open("capstone_streamlit.py", "w") as f:
    f.write(capstone_streamlit)

print("✅ capstone_streamlit.py written")
print()
print("IMPORTANT: Before running, open capstone_streamlit.py and:")
print("  1. Paste your DOCUMENTS list into the load_agent() function")
print("  2. Paste your CapstoneState TypedDict")
print("  3. Paste all your node functions")
print("  4. Paste the graph assembly code (graph = StateGraph(...) ...)")
print()
print("Then run: streamlit run capstone_streamlit.py")

---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** TODO — your name

**Domain chosen:** TODO

**What the agent does:** TODO — 2-3 sentences describing what problem the agent solves and who uses it.

**Knowledge base:** TODO — how many documents, what topics they cover.

**Tool used:** TODO — what tool you added and why it was useful for this domain.

**RAGAS baseline scores:**
- Faithfulness: TODO
- Answer Relevance: TODO
- Context Precision: TODO

**Test results:** TODO / 10 tests passed. Red-team: TODO / 2 passed.

**One thing I would improve with more time:** TODO — be specific. (e.g., "I would add a hybrid BM25 + vector search for better context precision" or "I would load real HR policy PDFs instead of hand-written summaries")

**Most surprising thing I learned building this:** TODO

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*